### Dependencias

In [13]:
import cv2
import os
import numpy as np
import mediapipe as mp
import tensorflow as tf
import time

from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.layers import Attention
from tensorflow.keras.layers import GlobalAveragePooling1D
from sklearn.model_selection import train_test_split

### Configuración

In [14]:
# DATA
DATASET_PATH = "dataset"
MODEL_PATH = "models"

# SIGNS TO COLLECT
SIGNS = ["Jesus", "Amor", "Familia"]

# SEQUENCE
SEQUENCE_LENGTH = 50
NUM_SEQUENCES = 50

# FEATURES
LANDMARKS = 21
COORDS = 3
BASE_FEATURES = LANDMARKS * COORDS
MAX_HANDS = 2
TOTAL_FEATURES = BASE_FEATURES * 2 * MAX_HANDS  # coords + velocity

### Inicio de MediaPipe

In [15]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

### Normalización

In [16]:
def normalize_landmarks(landmarks):

    coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks])

    wrist = coords[0]

    coords = coords - wrist

    scale = np.linalg.norm(coords[9])

    if scale != 0:
        coords = coords / scale

    return coords.flatten()

### Captura y vector final

In [17]:
# Movimiento entre Frames
def compute_velocity(curr, prev):

    if prev is None:
        return np.zeros_like(curr)

    return curr - prev

In [18]:
# Vector final
def extract_features(hand_landmarks, prev_features):

    coords = normalize_landmarks(hand_landmarks.landmark)

    velocity = compute_velocity(coords, prev_features)

    features = np.concatenate([coords, velocity])

    return features, coords

### Recolección de datos

In [19]:
def collect_dataset(sign):

    os.makedirs(f"{DATASET_PATH}/{sign}", exist_ok=True)

    cap = cv2.VideoCapture(0)

    sequence = []
    prev_features = None
    sequence_id = 0

    collecting = False
    cooldown = False
    cooldown_time = 2
    last_capture_time = 0

    print(f"\nRecolectando datos para: {sign}")
    print("Captura automática activada...")
    print("ESC para salir\n")

    while sequence_id < NUM_SEQUENCES:

        ret, frame = cap.read()
        if not ret:
            break

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(image)

        current_time = time.time()

        # 🔥 COOLDOWN
        if cooldown:
            if current_time - last_capture_time >= cooldown_time:
                cooldown = False
            else:
                cv2.putText(frame, "Cooldown...", (10, 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
                cv2.imshow("Capture", frame)
                if cv2.waitKey(10) & 0xFF == 27:
                    break
                continue

        # 🚀 INICIO AUTOMÁTICO
        if results.multi_hand_landmarks and not collecting:
            collecting = True
            sequence = []
            prev_features = None
            print(f"Iniciando secuencia {sequence_id}")

        # 📥 RECOLECCIÓN
        if collecting and results.multi_hand_landmarks:

            MAX_HANDS = 2

            # 🧠 Orden fijo: [Left, Right]
            hands_data = [None, None]

            if results.multi_handedness:
                for hand_landmarks, handedness in zip(
                    results.multi_hand_landmarks,
                    results.multi_handedness
                ):
                    label = handedness.classification[0].label

                    if label == "Left":
                        hands_data[0] = hand_landmarks
                    else:
                        hands_data[1] = hand_landmarks

            all_features = []
            current_coords = []

            for i in range(MAX_HANDS):

                if hands_data[i] is not None:

                    prev = prev_features[i] if prev_features is not None else None

                    features, coords = extract_features(hands_data[i], prev)

                else:
                    features = np.zeros(BASE_FEATURES * 2)
                    coords = np.zeros(BASE_FEATURES)

                all_features.append(features)
                current_coords.append(coords)

                # Dibujar cada mano
                if hands_data[i] is not None:
                    mp_drawing.draw_landmarks(
                        frame,
                        hands_data[i],
                        mp_hands.HAND_CONNECTIONS
                    )

            final_features = np.concatenate(all_features)

            prev_features = current_coords
            sequence.append(final_features)

            # ✅ Guardar secuencia completa
            if len(sequence) == SEQUENCE_LENGTH:

                np.save(
                    f"{DATASET_PATH}/{sign}/{sequence_id}.npy",
                    np.array(sequence, dtype=np.float32)
                )

                print(f"✅ Secuencia guardada: {sequence_id}")

                sequence_id += 1
                collecting = False
                cooldown = True
                last_capture_time = current_time

                sequence = []
                prev_features = None

        else:
            if collecting:
                print("⚠️ Mano perdida, secuencia descartada")

            collecting = False
            sequence = []
            prev_features = None

        # 🖥️ UI
        cv2.putText(frame, f"Sign: {sign} | Seq: {sequence_id}/{NUM_SEQUENCES}",
                    (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

        cv2.putText(frame, f"Frames: {len(sequence)}/{SEQUENCE_LENGTH}",
                    (10,60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)

        status = "Collecting" if collecting else "Waiting"
        cv2.putText(frame, f"Status: {status}",
                    (10,120), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        cv2.imshow("Capture", frame)

        if cv2.waitKey(10) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()

In [20]:
for sign in SIGNS:
    collect_dataset(sign)


Recolectando datos para: Jesus
Captura automática activada...
ESC para salir

Iniciando secuencia 0
✅ Secuencia guardada: 0
Iniciando secuencia 1
✅ Secuencia guardada: 1
Iniciando secuencia 2
✅ Secuencia guardada: 2
Iniciando secuencia 3
✅ Secuencia guardada: 3
Iniciando secuencia 4
✅ Secuencia guardada: 4
Iniciando secuencia 5
✅ Secuencia guardada: 5
Iniciando secuencia 6
✅ Secuencia guardada: 6
Iniciando secuencia 7
✅ Secuencia guardada: 7
Iniciando secuencia 8
✅ Secuencia guardada: 8
Iniciando secuencia 9
✅ Secuencia guardada: 9

Recolectando datos para: Amor
Captura automática activada...
ESC para salir

Iniciando secuencia 0
✅ Secuencia guardada: 0
Iniciando secuencia 1
✅ Secuencia guardada: 1
Iniciando secuencia 2
✅ Secuencia guardada: 2
Iniciando secuencia 3
✅ Secuencia guardada: 3
Iniciando secuencia 4
✅ Secuencia guardada: 4
Iniciando secuencia 5
✅ Secuencia guardada: 5
Iniciando secuencia 6
✅ Secuencia guardada: 6
Iniciando secuencia 7
✅ Secuencia guardada: 7
Iniciando secue

### Crear Dataset

In [40]:
def build_dataset():

    X = []
    y = [] 

    labels = os.listdir(DATASET_PATH)
    label_map = {label: i for i, label in enumerate(labels)}

    for label in labels:

        for file in os.listdir(f"{DATASET_PATH}/{label}"):

            seq = np.load(f"{DATASET_PATH}/{label}/{file}")

            # 🔥 Forzar tipo y shape
            try:
                seq = np.array(seq, dtype=np.float32)

                if seq.shape != (SEQUENCE_LENGTH, TOTAL_FEATURES):
                    print("❌ Eliminando:", file, seq.shape)
                    continue

                X.append(seq)
                y.append(label_map[label])

            except Exception as e:
                print("💥 Error en archivo:", file, e)

    # 🔥 FORZAR conversión real
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int32)

    return X, y, label_map

### Crear Modelo .Keras

In [41]:
def create_model(num_classes):

    inputs = tf.keras.Input(shape=(SEQUENCE_LENGTH, TOTAL_FEATURES))

    x = LSTM(128, return_sequences=True)(inputs)
    x = Dropout(0.3)(x)

    x = LSTM(128, return_sequences=False)(x)

    x = Dense(64, activation="relu")(x)

    outputs = Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

### Training

In [42]:
X, y, label_map = build_dataset()

print("X shape:", X.shape)
print("y shape:", y.shape)
print(type(X))
print(X.dtype)

print("Primer elemento tipo:", type(X[0]))
print("Shape elemento 0:", X[0].shape)

# Buscar errores ocultos
for i, seq in enumerate(X):
    if not isinstance(seq, np.ndarray):
        print("❌ No es ndarray:", i)
    elif seq.shape != (50, 252):
        print("❌ Shape raro:", i, seq.shape)
    elif seq.dtype != np.float32:
        print("⚠️ dtype raro:", i, seq.dtype)

X shape: (30, 50, 252)
y shape: (30,)
<class 'numpy.ndarray'>
float32
Primer elemento tipo: <class 'numpy.ndarray'>
Shape elemento 0: (50, 252)


In [43]:
model = create_model(len(label_map))

# Separar datos
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Entrenar
model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32
)
#Guardar modelo
os.makedirs(MODEL_PATH, exist_ok=True)

model.save(f"{MODEL_PATH}/sign_model.keras")

Epoch 1/50
1/1 [==============================] - 5s 5s/step - loss: 1.1064 - accuracy: 0.3333 - val_loss: 0.8209 - val_accuracy: 1.0000
Epoch 2/50
1/1 [==============================] - 0s 111ms/step - loss: 0.8405 - accuracy: 1.0000 - val_loss: 0.5924 - val_accuracy: 1.0000
Epoch 3/50
1/1 [==============================] - 0s 93ms/step - loss: 0.6177 - accuracy: 1.0000 - val_loss: 0.4234 - val_accuracy: 1.0000
Epoch 4/50
1/1 [==============================] - 0s 100ms/step - loss: 0.4472 - accuracy: 1.0000 - val_loss: 0.2842 - val_accuracy: 1.0000
Epoch 5/50
1/1 [==============================] - 0s 105ms/step - loss: 0.3063 - accuracy: 1.0000 - val_loss: 0.1819 - val_accuracy: 1.0000
Epoch 6/50
1/1 [==============================] - 0s 95ms/step - loss: 0.1946 - accuracy: 1.0000 - val_loss: 0.1093 - val_accuracy: 1.0000
Epoch 7/50
1/1 [==============================] - 0s 97ms/step - loss: 0.1183 - accuracy: 1.0000 - val_loss: 0.0650 - val_accuracy: 1.0000
Epoch 8/50
1/1 [==========

### Exportación

In [44]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()

with open(f"{MODEL_PATH}/sign_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved")

INFO:tensorflow:Assets written to: C:\Users\ledys\AppData\Local\Temp\tmpt7jltdlx\assets


INFO:tensorflow:Assets written to: C:\Users\ledys\AppData\Local\Temp\tmpt7jltdlx\assets


TFLite model saved


### Test Local

In [45]:
sequence = []
prev_features = None

cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()
    if not ret:
        break

    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(image)

    if results.multi_hand_landmarks:

        MAX_HANDS = 2

        # 🧠 Ordenar manos: [Left, Right]
        hands_data = [None, None]

        if results.multi_handedness:
            for hand_landmarks, handedness in zip(
                results.multi_hand_landmarks,
                results.multi_handedness
            ):
                label = handedness.classification[0].label

                if label == "Left":
                    hands_data[0] = hand_landmarks
                else:
                    hands_data[1] = hand_landmarks

        all_features = []
        current_coords = []

        for i in range(MAX_HANDS):

            if hands_data[i] is not None:

                hand_landmarks = hands_data[i]

                prev = prev_features[i] if prev_features is not None else None

                features, coords = extract_features(hand_landmarks, prev)

            else:
                features = np.zeros(BASE_FEATURES * 2)
                coords = np.zeros(BASE_FEATURES)

            current_coords.append(coords)
            all_features.append(features)

            # Dibujar cada mano
            if hands_data[i] is not None:
                mp_drawing.draw_landmarks(
                    frame,
                    hands_data[i],
                    mp_hands.HAND_CONNECTIONS
                )

        final_features = np.concatenate(all_features)

        prev_features = current_coords
        sequence.append(final_features)

        sequence = sequence[-SEQUENCE_LENGTH:]

        if len(sequence) == SEQUENCE_LENGTH:

            input_data = np.expand_dims(sequence, axis=0)

            prediction = model.predict(input_data, verbose=0)

            class_id = np.argmax(prediction)
            confidence = np.max(prediction)

            if confidence > 0.8:

                sign = list(label_map.keys())[class_id]

                cv2.putText(
                    frame,
                    f"{sign} {confidence:.2f}",
                    (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 255, 0),
                    2
                )

    else:
        # ❗ Reset si no hay manos
        prev_features = None
        sequence = []

    cv2.imshow("Inference", frame)

    if cv2.waitKey(10) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()